# Table Transformer v1.1-all — DIMER table structure recognition tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/table-transformer-structure-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/table-transformer-structure-pipeline/blob/main/tutorials/table_transformer_structure_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2Ftable--transformer--structure--recognition--v1.1--all-ffcc4d?style=flat)](https://huggingface.co/microsoft/table-transformer-structure-recognition-v1.1-all) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2Ftable--transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/table-transformer) [![arXiv](https://img.shields.io/badge/arXiv-2303.00716-b31b1b.svg)](https://arxiv.org/abs/2303.00716)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** table structure recognition on a table-crop image (boxes labelled `table`, `table column`, `table row`, `table column header`, `table projected row header`, `table spanning cell`) using the pinned `microsoft/table-transformer-structure-recognition-v1.1-all` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/table_transformer_structure_pipeline/pipeline.py` at revision `f692c60a1179`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `7587a7ef111d9dcbf8ac695f1376ab7014340a0c` (~116 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the DETR-style model reads one table image resized so its longest edge is 800 px, runs an in-library ResNet-18 backbone and a 6-layer encoder–decoder, and emits exactly 125 query proposals, each a box and a softmax over the six structure classes and *no object*; the processor keeps the queries whose class score reaches the threshold and maps their boxes back to input pixels. The rows and columns are the load-bearing output: intersecting them yields the cell grid, and the header, projected-row-header and spanning-cell boxes refine it. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and image-processor configuration, and the carried module adds snapshot verification, the input contract, a fixed output contract and the `box_iou`, `structure_summary`, `validate_inputs` and `evaluation_report` helpers. The default sample is a table rendered in code and cropped with the upstream script's 10 px padding, whose drawn row/column boxes serve as references; its `box_iou` values are demonstration (plumbing) evidence for one table, not a recognition benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, render a synthetic table crop with reference structure boxes (or upload your own table image) and validate it into an input manifest, run the supported task, read the six structure classes, the class scores and the caller-owned threshold correctly, derive the implied cell grid with `structure_summary`, exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with per-label `box_iou` only when reference boxes exist and `not-measurable` otherwise, and export machine-readable structure objects plus an annotated image and provenance.

**This notebook does not demonstrate:** table *detection* on a full page (the sibling `table-transformer-detection-pipeline` finds the crop this notebook expects), OCR or cell text extraction, assembling the final cell grid or HTML/CSV export (the rows and columns are returned; intersecting them is left to the caller), precision/recall or GriTS evaluation (which needs a labelled table set), or any training. The model was trained on PubTables-1M and FinTabNet.c renders; scans, photographs and non-Latin layouts are outside what this notebook measures. A non-table image still yields structure objects.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 4.8 s to load and 0.12 s per `recognize` on the 730×350 synthetic crop in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 115 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union measures; that a table's cell grid is the intersection of its row and column boxes.
- **Data:** the default sample is a deterministic 730×350 crop of an 8×5 ruled table rendered in code with Pillow's bundled font, with 10 px of white page around it (the upstream inference script's crop padding), so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar) showing a **single table**, ideally cropped with a little page margin, any colour mode, sides between 16 and 4096 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/table-transformer-structure-recognition-v1.1-all` snapshot (~116 MB in total) at revision `7587a7ef111d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'table-transformer-structure-pipeline',
    'repository_revision': 'f692c60a1179ec6cd7902357e15c2125d44e01c0',
    'embedded_module': 'src/table_transformer_structure_pipeline/pipeline.py',
    'embedded_modules': ['src/table_transformer_structure_pipeline/pipeline.py'],
    'module_sha256': 'c6b84f8ec228d883c3a099e5fa0a6d26f44d1aae715d194515bdcceac818cd6b',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/table_transformer_structure_pipeline/` @ `f692c60a1179`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/table_transformer_structure_pipeline/pipeline.py`

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "microsoft/table-transformer-structure-recognition-v1.1-all"
MODEL_REVISION = "7587a7ef111d9dcbf8ac695f1376ab7014340a0c"
MODEL_LICENSE = "mit"
MODEL_KEY = "table-transformer-structure-v1.1-all"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The six structure classes the checkpoint was fine-tuned on (config.json id2label, in id order).
LABELS = (
    "table",
    "table column",
    "table row",
    "table column header",
    "table projected row header",
    "table spanning cell",
)
# Recognition threshold: the value the upstream repository's inference script applies to every
# structure class (microsoft/table-transformer src/inference.py `structure_class_thresholds`, main @
# 16d124f, 2023-09-07). It gates a softmax class score over 125 DETR queries that was not calibrated
# for any document domain; the deployment owns tuning it on labelled tables.
RECOGNITION_THRESHOLD = 0.5
# The upstream inference script crops each detected table with this many pixels of padding before
# structure recognition; the tutorial reproduces that convention. It is documentation, not enforced.
UPSTREAM_CROP_PADDING = 10
# The checkpoint's DETR decoder emits exactly num_queries proposals per image (config.json), so no
# image can yield more than this many structure objects.
MAX_DETECTIONS = 125
# Input ceilings. The processor resizes so the longest edge is 800 px (preprocessor_config.json
# `size.longest_edge`), so image cost is bounded whatever the caller sends; the side ceiling only
# guards memory during decoding and resizing.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# preprocessor_config.json says `size: {"longest_edge": 800}`; the pinned transformers image processor
# only accepts the two-key form, and capping both edges at 800 resizes every image so its longest edge
# is 800 px — the same transform (see from_pretrained).
PROCESSOR_SIZE = {"shortest_edge": 800, "longest_edge": 800}


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for any caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"threshold must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one table image as PIL.Image.Image (any mode, converted to RGB): a crop of a single table, "
        f"ideally with about {UPSTREAM_CROP_PADDING} px of page around it, as the upstream inference "
        "script crops tables"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB; the processor resizes so the longest edge is 800 px, normalises with "
        "ImageNet mean/std, and returned boxes are mapped back to input pixels"
    ),
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``recognize`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = RECOGNITION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``recognize`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (recognize takes one table image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def structure_summary(result: Mapping[str, Any]) -> dict[str, Any]:
    """Count the structure objects per label and derive the grid size the rows and columns imply.

    ``n_rows`` x ``n_columns`` is the cell grid a caller would build by intersecting row and column
    boxes (the upstream ``objects_to_structures`` step); this helper only counts, it does not build cells.
    """
    counts = {label: 0 for label in LABELS}
    for det in result["detections"]:
        counts[det["label"]] += 1
    return {
        "counts": counts,
        "n_rows": counts["table row"],
        "n_columns": counts["table column"],
        "n_cells_implied": counts["table row"] * counts["table column"],
        "has_column_header": counts["table column header"] > 0,
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[Sequence[float]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (label -> xyxy reference boxes for that structure class) the report
    carries one ``box_iou`` entry per reference — the best-overlapping detection **of the same label**
    — plus per-label reference/detection counts, as sample-sanity geometry evidence; without them the
    verdict is ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    detections = list(result["detections"])
    summary = structure_summary(result)
    base = {
        "task": "table structure recognition on a table-crop image",
        "decision_rule": (
            "a DETR query survives when its softmax score for one of the six structure classes reaches "
            "the threshold; the score is a class probability under the model's own softmax, not a "
            "calibrated estimate for the deployment's tables"
        ),
        "threshold": result.get("threshold", RECOGNITION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "structure_summary": summary,
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth structure boxes were supplied for the evaluated table",
            "needs": (
                "labelled row/column/header/spanning-cell boxes on your own tables, scored per object with "
                "box_iou and aggregated into per-class precision/recall or the cell-level metrics (GriTS) "
                "the upstream paper uses; no such labelled set ships with this repository"
            ),
        }
    metrics = []
    for label, boxes in ground_truth_boxes.items():
        if label not in LABELS:
            raise ValueError(f"unknown reference label {label!r}; expected one of {LABELS}")
        same = [det for det in detections if det["label"] == label]
        for index, box in enumerate(boxes):
            ious = [box_iou(det["box"], box) for det in same]
            best = max(range(len(ious)), key=ious.__getitem__) if ious else None
            metrics.append(
                {
                    "id": "box_iou",
                    "label": label,
                    "reference": f"{label}-{index}",
                    "value": ious[best] if best is not None else 0.0,
                    "n_reference": len(boxes),
                    "n_detected": len(same),
                    "estimation": (
                        "one reference box per structure object on a single table, no dispersion estimate"
                    ),
                }
            )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial table; geometry sanity evidence, "
            "not a structure-recognition benchmark"
        ),
        "needs": (
            "a labelled table set from the deployment domain (publishers, scans, layouts) for any "
            "precision/recall or GriTS claim"
        ),
    }


@dataclass
class TableTransformerStructurePipeline:
    """Table structure recognition (rows, columns, headers, spanning cells) over the pinned checkpoint."""

    _runner: Callable[[Image.Image, float], list[dict[str, Any]]]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TableTransformerStructurePipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoImageProcessor, TableTransformerForObjectDetection

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        # The pinned preprocessor_config.json declares `size: {"longest_edge": 800}`, a shape the pinned
        # transformers release's DetrImageProcessor refuses ("Size must contain ... 'shortest_edge' and
        # 'longest_edge'"). PROCESSOR_SIZE is the same resize expressed in the accepted form: with both
        # edges capped at 800 the longest edge always lands on 800 px, exactly as the upstream file says.
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, size=dict(PROCESSOR_SIZE), **kwargs
        )
        # This checkpoint's config carries an in-library ResNet backbone_config (use_timm_backbone=False),
        # so nothing is fetched at construction; use_pretrained_backbone=False is passed anyway so the
        # loader can never reach for ImageNet weights the checkpoint already contains.
        model = TableTransformerForObjectDetection.from_pretrained(
            source,
            revision=MODEL_REVISION,
            trust_remote_code=False,
            use_pretrained_backbone=False,
            **kwargs,
        )
        model = model.to(resolved_device).eval()
        id2label = {int(k): v for k, v in model.config.id2label.items()}

        def runner(image: Image.Image, threshold: float) -> list[dict]:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs)
            result = processor.post_process_object_detection(
                outputs, threshold=threshold, target_sizes=[image.size[::-1]]
            )[0]
            return [
                {
                    "box": [float(v) for v in box.tolist()],
                    "label": id2label[int(label)],
                    "score": float(score),
                }
                for box, label, score in zip(result["boxes"], result["labels"], result["scores"], strict=True)
            ]

        return cls(runner, resolved_device)

    def recognize(self, image: Image.Image, *, threshold: float = RECOGNITION_THRESHOLD) -> dict[str, Any]:
        """Recognise the structure of one table-crop image; boxes are xyxy pixel coordinates in the input."""
        rgb, checked = _check_inputs(image, threshold)
        detections = self._runner(rgb, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > num_queries {MAX_DETECTIONS}"
            )
        for det in detections:
            if set(det) != {"box", "label", "score"} or len(det["box"]) != 4 or det["label"] not in LABELS:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `7587a7ef111d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TableTransformerStructurePipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "table-transformer-structure-v1.1-all",
  "modelId": "microsoft/table-transformer-structure-recognition-v1.1-all",
  "revision": "7587a7ef111d9dcbf8ac695f1376ab7014340a0c",
  "files": [
    {
      "path": "README.md",
      "bytes": 1056,
      "sha256": "01eed360e0f54ad297ab347cfef208d68d8ac2c108b808147bb7af9f8d9132c8"
    },
    {
      "path": "config.json",
      "bytes": 76761,
      "sha256": "17a8a6edfb9e394263fa6ba9b82176ebccdfcc5d6cd29121ec91572c7d6be22c"
    },
    {
      "path": "model.safetensors",
      "bytes": 115437156,
      "sha256": "9df416575a3a36ebd0129342d4f597f14d6e5170268f3d52d28584ab4466a501"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 374,
      "sha256": "eead409bb80e36ae85b8377642c54550f0504f65688ba3a4967950cafe461df2"
    }
  ],
  "totalBytes": 115515347
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TableTransformerStructurePipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Render the synthetic table crop or optional BYOD

The default sample is **synthetic** and carries its own reference boxes: an 8×5 ruled table (a header row and seven data rows of short tokens, rendered with Pillow's bundled font) is drawn on a white page at `[70, 330, 780, 660]` and cropped with `UPSTREAM_CROP_PADDING` (10) px of page around it, the way the upstream inference script crops each detected table before structure recognition; the crop is 730×350. This is the same crop the repository's smoke run used. The drawn table, row, column and column-header boxes (in crop coordinates) are the references for the per-label `box_iou` sanity check later; they are not a labelled dataset, so nothing here is a precision/recall measurement. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one table image — no reference boxes exist for it, so the evaluation report will be `not-measurable`.

The recognition threshold is a **caller-owned request parameter**, not a pipeline constant: a query survives when its softmax score for one of the six structure classes reaches it. The package default (`RECOGNITION_THRESHOLD = 0.5`) is the value the upstream repository's inference script applies to every structure class, not a calibration; it is exposed here as a form parameter and passed explicitly on every call. Nothing is validated in this cell — the next section hands the image and the threshold to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the crop size and digest, the threshold, and the number of reference boxes per label.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
threshold = 0.5  # @param {type:"number"}


def synthetic_table(rows=8, cols=5, x0=70, y0=330, x1=780, y1=660, pad=UPSTREAM_CROP_PADDING):
    """An 8x5 ruled table rendered with Pillow's bundled font, cropped with `pad` px of page around it."""
    page = Image.new('RGB', (850, 1100), 'white')
    d = ImageDraw.Draw(page)
    body = ImageFont.load_default(size=15)
    d.rectangle([x0, y0, x1, y1], outline='black', width=2)
    rh, cw = (y1 - y0) / rows, (x1 - x0) / cols
    d.line([(x0, y0 + rh), (x1, y0 + rh)], fill='black', width=2)
    for r in range(2, rows):
        d.line([(x0, y0 + rh * r), (x1, y0 + rh * r)], fill=(120, 120, 120), width=1)
    for c in range(1, cols):
        d.line([(x0 + cw * c, y0), (x0 + cw * c, y1)], fill=(120, 120, 120), width=1)
    for r in range(rows):
        for c in range(cols):
            token = ('Region' if c == 0 else f'Q{c}') if r == 0 else (f'North {r}' if c == 0 else f'{(r * 7 + c * 13) % 97 + 1},{(r * 31 + c) % 900 + 100:03d}')
            d.text((x0 + cw * c + 8, y0 + rh * r + rh / 2 - 8), token, fill='black', font=body)
    crop = page.crop((x0 - pad, y0 - pad, x1 + pad, y1 + pad))
    ox, oy = x0 - pad, y0 - pad
    refs = {
        'table': [[x0 - ox, y0 - oy, x1 - ox, y1 - oy]],
        'table row': [[x0 - ox, y0 + rh * r - oy, x1 - ox, y0 + rh * (r + 1) - oy] for r in range(rows)],
        'table column': [[x0 + cw * c - ox, y0 - oy, x0 + cw * (c + 1) - ox, y1 - oy] for c in range(cols)],
        'table column header': [[x0 - ox, y0 - oy, x1 - ox, y0 + rh - oy]],
    }
    return crop, {label: [[float(v) for v in box] for box in boxes] for label, boxes in refs.items()}


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    drawn_boxes = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic table: no randomness, so no seed is needed and the digest is stable per Pillow build.
    image, drawn_boxes = synthetic_table()
    image_name = 'synthetic_table_crop_730x350.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'threshold': threshold, 'reference_boxes': None if drawn_boxes is None else {k: len(v) for k, v in drawn_boxes.items()}})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `recognize` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px and a threshold in `[0, 1]` — and returns an **input manifest** naming the schema (including the six labels and the 125-query ceiling on structure objects), the input's observed mode and size, the threshold, and the verdict. The manifest is written to `outputs/table_transformer_structure_input_manifest.json`. To show what rejection looks like, the cell also validates a threshold outside `[0, 1]` and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and resized by the processor so its longest edge is 800 px; boxes are mapped back to input pixels, and nothing else is dropped or altered. The pipeline cannot tell whether the image is a single table: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_DETECTIONS': MAX_DETECTIONS, 'LABELS': list(LABELS), 'RECOGNITION_THRESHOLD': RECOGNITION_THRESHOLD, 'UPSTREAM_CROP_PADDING': UPSTREAM_CROP_PADDING}})
input_manifest = validate_inputs(image, threshold=threshold, names=[image_name])
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(image, threshold=1.5)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'out-of-range-threshold-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/table_transformer_structure_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Recognise the structure and read the scores correctly

`recognize` returns a dict with `detections` — a list of `{box, label, score}` **ordered by descending score**, `box` in xyxy pixel coordinates of the input, `label` one of the six structure classes — plus the threshold used, `width`, `height` and the model identity. At most 125 objects can ever be returned (the DETR decoder has 125 queries). Each `score` is the query's **softmax class probability under the model's own head, not a calibrated estimate for your tables**. The threshold you passed is the only decision rule; the pipeline ships 0.5 as a default (the upstream inference script's per-class value), not as a calibration, and the caller owns it per deployment. `structure_summary` counts the objects per label and reports the cell grid the rows and columns imply (`n_rows` × `n_columns`); building the actual cells by intersecting the boxes is the caller's next step. Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move scores in the third or fourth decimal place. As recorded in the model card, the repository's CPU smoke on this same crop at threshold 0.5 returned exactly 1 `table`, 5 `table column`, 8 `table row` and 1 `table column header` (15 objects, every score 1.00) — a count that matches the rendered grid — and the same 15 at 0.9; that is one observation on one synthetic table, not a calibration point.

In [ ]:
result = pipe.recognize(image, threshold=threshold)
summary = structure_summary(result)
print({'n_detections': len(result['detections']), 'threshold': result['threshold'], 'device': pipe.device, 'summary': summary})
for rank, det in enumerate(result['detections'], start=1):
    print(f"{rank:>3}. score {det['score']:.4f}  label {det['label']!r:28}  box {[round(v, 1) for v in det['box']]}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No recognition metric is reported by default: per-class precision/recall or the cell-level GriTS scores the upstream paper uses need a labelled table set, and this repository ships none. The repository's only metric helper is `box_iou(a, b)`; when reference boxes are supplied, keyed by label, the report carries one `box_iou` entry per reference — matched only against detections **of the same label** — together with the reference and detected counts for that label, with the verdict `sample-sanity`. On the synthetic path those references are rows and columns **you rendered yourself**, so a high IoU proves only that the input contract, forward pass and coordinate mapping round-trip. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/table_transformer_structure_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, drawn_boxes, sample_kind=sample_kind)
with open('outputs/table_transformer_structure_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k != 'metrics'}, indent=2))
for metric in report['metrics']:
    print(f"{metric['reference']:28} iou {metric['value']:.3f}  (references {metric['n_reference']}, detected {metric['n_detected']})")
if report['verdict'] == 'not-measurable':
    print('No reference boxes exist for this input, so box_iou is not computed; inspect the annotated PNG instead.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (score-ordered structure objects with boxes and labels, the threshold), the structure summary, the evaluation report, the input manifest, the sample identity, digest and reference boxes, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The objects are also written as CSV with explicit `image`, `rank`, `label`, `score`, `x0`, `y0`, `x1`, `y1` columns so score ordering survives downstream use, and an annotated PNG draws rows in blue, columns in green and everything else in red for visual inspection (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import csv

COLOURS = {'table row': (40, 90, 220), 'table column': (0, 160, 0)}
annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for det in result['detections']:
    draw.rectangle(det['box'], outline=COLOURS.get(det['label'], (200, 30, 30)), width=2)
annotated.save('outputs/table_transformer_structure_annotated.png')
payload = {
    'prediction': result,
    'structure_summary': summary,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'reference_boxes': drawn_boxes},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/table_transformer_structure_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/table_transformer_structure_objects.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1'])
    for rank, det in enumerate(result['detections'], start=1):
        writer.writerow([image_name, rank, det['label'], f"{det['score']:.6f}", *[f"{v:.2f}" for v in det['box']]])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The boxes are the structure objects the model sees in an image it assumes to be one table; the six labels are the model's vocabulary, the softmax score is not calibrated for your documents, and the threshold is a request parameter you own (the default is the upstream script's value, not a tuned operating point). On the synthetic crop the per-label `box_iou` values in the evaluation report compare objects to rows and columns you rendered yourself and the verdict is `sample-sanity`, which proves only that the input contract, forward pass and coordinate mapping work; they say nothing about scans, borderless or merged-cell tables, rotated tables, multi-line cells, or non-Latin documents, and a BYOD result is a single-table observation with the verdict `not-measurable`. **The model emits structure objects for any image**: the repository's smoke run fed it a 4096×4096 blank image and got 23 objects above 0.5, so a crop that is not a table produces confident nonsense rather than an empty result — detect first, then recognise. The pipeline provides no page-level detection, no OCR, no cell assembly, no GriTS evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** raise `threshold` to 0.9 and check the object count stays at 15 (the smoke run found it did); merge two header cells in `synthetic_table` and look for a `table spanning cell`; enable `USE_BYOD` with a real table crop, hand-label its rows and columns and pass them to `evaluation_report` to see the verdict switch to `sample-sanity`; then intersect the row and column boxes to build the cell grid and compare it with the `n_cells_implied` count.

## References

- Repository README: https://github.com/kurtvalcorza/table-transformer-structure-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/table-transformer-structure-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/table-transformer-structure-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/microsoft/table-transformer-structure-recognition-v1.1-all
- Upstream code: https://github.com/microsoft/table-transformer
- Aligning benchmark datasets for table structure recognition (Smock, Pesala, Abraham, 2023): https://arxiv.org/abs/2303.00716
- PubTables-1M (Smock, Pesala, Abraham, 2021): https://arxiv.org/abs/2110.00061